In [41]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# 1. Загрузка
train_df = pd.read_csv(r'C:\Users\IVAN\Desktop\ML обучение с Gemini\Файлы\Titanic - Machine Learning from Disaster\train.csv')
test_df = pd.read_csv(r'C:\Users\IVAN\Desktop\ML обучение с Gemini\Файлы\Titanic - Machine Learning from Disaster\test.csv')

test_ids = test_df['PassengerId']

# 2. Предобработка
def preprocess_data(train, test):
    df = pd.concat([train.drop('Survived', axis=1), test], axis=0).reset_index(drop=True)
    
    # Заполнение пропусков медианами и модой
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    
    # Кодирование категориальных признаков в числа
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df = pd.get_dummies(df, columns=['Embarked'], drop_first=True, dtype=int)
    
    # Удаляем нечисловые/сложные колонки
    drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
    df = df.drop(columns=drop_cols)
    
    # Разделяем обратно на train и test
    X_tr_raw = df.iloc[:len(train)].copy()
    X_te_raw = df.iloc[len(train):].copy()
    
    # Масштабирование признаков (StandardScaler)
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr_raw)
    X_te_scaled = scaler.transform(X_te_raw)
    
    return X_tr_scaled, X_te_scaled

# Готовые матрицы для твоей модели:
X_train, X_test = preprocess_data(train_df, test_df)
y_train = train_df['Survived'].values

print("Данные готовы!")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

Данные готовы!
X_train shape: (891, 8)
X_test shape: (418, 8)


In [42]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

param_grid = [
    {
        'solver': ['lbfgs'],
        'penalty': ['l2'],
        'C': np.logspace(-3, 3, 20),
        'max_iter': [1000]
    },
    {
        'solver': ['liblinear'],
        'penalty': ['l1', 'l2'],
        'C': np.logspace(-3, 3, 10),
        'max_iter': [1000]
    }
]

grid_search = GridSearchCV(
    estimator=LogisticRegression(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f"Лучшая Accuracy на кросс-валидации: {grid_search.best_score_:.4f}")
print(f"Лучшие параметры: {grid_search.best_params_}")

best_model = grid_search.best_estimator_
prediction = best_model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived': prediction
})
submission.to_csv(
    r'C:\Users\IVAN\Desktop\ML обучение с Gemini\Файлы\Titanic - Machine Learning from Disaster\submission.csv', 
    index=False
    )

Лучшая Accuracy на кросс-валидации: 0.8002
Лучшие параметры: {'C': np.float64(0.008858667904100823), 'max_iter': 1000, 'penalty': 'l2', 'solver': 'lbfgs'}


c:\Users\IVAN\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
